In [ ]:
import os
import sys
from google.colab import userdata

# =====================================================================
# 1. SECURE CREDENTIALS & CONSTANTS
# =====================================================================
TOKEN = userdata.get('GithubToken')
USERNAME = "mervynzwkoh"
REPO = "gout-transcriptome-causal"

# Ensure we are operating from the default Colab root directory
%cd /content/

# =====================================================================
# 2. CLONE & INITIALIZE YOUR WORKSPACE
# =====================================================================
print("🗂️ Setting up your personal repository...")
if not os.path.exists(f"/content/{REPO}"):
    !git clone https://{TOKEN}@github.com/{USERNAME}/{REPO}.git
else:
    print("-> Personal repository folder already exists. Skipping clone.")

# Move permanently into your project folder and switch to your feature branch
%cd /content/{REPO}
!git checkout feature/model-fine-tuning
!git pull origin feature/model-fine-tuning

# =====================================================================
# 3. CLONE BIONEMO FRAMEWORK (OUTSIDE YOUR WORKSPACE)
# =====================================================================
print("\n🧬 Fetching external BioNeMo Framework...")
%cd /content/
if not os.path.exists("/content/bionemo-framework"):
    !git clone https://github.com/NVIDIA/bionemo-framework.git
else:
    print("-> BioNeMo framework folder already exists. Skipping clone.")

# =====================================================================
# 4. MODULAR SUB-PACKAGE INSTALLATION
# =====================================================================
print("\n📦 Installing single-cell framework dependencies...")
# Install foundational interfaces and single-cell data loading utilities
!pip install -q -e /content/bionemo-framework/sub-packages/bionemo-core
!pip install -q -e /content/bionemo-framework/sub-packages/bionemo-scdl
!pip install --no-build-isolation transformer_engine[pytorch]

# =====================================================================
# 5. BRIDGE SYSTEM PATH FOR RECIPES
# =====================================================================
# Tell Python's engine exactly where the Geneformer model recipes live
recipe_path = "/content/bionemo-framework/bionemo-recipes"
if recipe_path not in sys.path:
    sys.path.append(recipe_path)

# Return back to your own repository so file saves land in the right spot
%cd /content/{REPO}
print("\n🚀 Setup complete! Workspace and BioNeMo modules are fully synced.")

In [ ]:
REPO = "gout-transcriptome-causal"
%cd /content/{REPO}

In [ ]:
import os
# Force CUDA to execute commands synchronously so errors report on the exact line
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

import sys
import torch
import torch.nn as nn

# 1. Direct path injection to the native TransformerEngine recipe
recipe_dir = "/content/bionemo-framework/bionemo-recipes/models/geneformer/src/geneformer"
if recipe_dir not in sys.path:
    sys.path.append(recipe_dir)

# 2. Import the raw accelerated building blocks
from modeling_bert_te import BertModel, TEBertConfig

# 3. Custom Modular Regression Wrapper
class GeneformerForRegression(nn.Module):
    def __init__(self, config):
        super().__init__()
        # Initialize NVIDIA's optimized TransformerEngine backbone
        self.bert = BertModel(config)

        # Append your Regression Head (768 features -> 1 continuous dosage prediction)
        self.regression_head = nn.Linear(config.hidden_size, 1)

    def forward(self, input_ids, attention_mask=None):
        # 1. Create a default attention mask if none is provided
        # This satisfies TransformerEngine's strict internal routing conditions
        if attention_mask is None:
            attention_mask = torch.ones_like(input_ids)

        # 2. Extract hidden states directly by running input_ids through the model
        # Passing an explicit mask prevents the model from looking for 'get_head_mask'
        hidden_states = self.bert(input_ids=input_ids, attention_mask=attention_mask)

        # Handle outputs safely depending on the return layout (Tensor vs. Tuple)
        if isinstance(hidden_states, tuple):
            hidden_states = hidden_states[0]

        # 3. Extract the very first token ([CLS] token placeholder) which acts
        # as the high-dimensional representation of the entire cell state
        cell_embedding = hidden_states[:, 0, :]

        # Predict the continuous dosage proxy (num_features)
        dosage_prediction = self.regression_head(cell_embedding)
        return dosage_prediction

print("Custom GeneformerForRegression architecture wrapper successfully defined.")

In [ ]:
import sys
import torch
import torch.nn as nn

# Link to native recipes
recipe_dir = "/content/bionemo-framework/bionemo-recipes/models/geneformer/src/geneformer"
if recipe_dir not in sys.path:
    sys.path.append(recipe_dir)

from modeling_bert_te import BertModel, TEBertConfig

# =====================================================================
# 1. INITIALIZE ARCHITECTURE CONFIGURATION
# =====================================================================
model_config = TEBertConfig(
    hidden_size=768,
    num_attention_heads=12,
    num_hidden_layers=6,
    vocab_size=40000,
    max_position_embeddings=2048
)
model_config.position_embedding_type = "absolute"
model_config.is_decoder = False

# =====================================================================
# 2. THE UNROLLED ARCHITECTURE WRAPPER
# =====================================================================
class GeneformerForRegression(nn.Module):
    def __init__(self, config):
        super().__init__()
        # Load the base model (but we will NOT use its default forward wrapper)
        self.bert = BertModel(config)
        self.regression_head = nn.Linear(config.hidden_size, 1)

    def forward(self, input_ids):
        # 1. Step inside the model: Get initial embeddings directly
        hidden_states = self.bert.embeddings(input_ids=input_ids)

        # 2. Construct the exact hardware-level Attention Mask shape (Batch, 1, 1, Seq)
        # In log-space math, 0.0 means "pay attention", -10000.0 means "ignore"
        batch_size, seq_length = input_ids.shape
        attention_mask = torch.zeros(
            (batch_size, 1, 1, seq_length),
            dtype=hidden_states.dtype,
            device=hidden_states.device
        )

        # 3. THE UNROLLING BYPASS: Execute TransformerEngine layers manually
        # This completely dodges Hugging Face's 8-argument interface mismatch
        for layer_module in self.bert.encoder.layer:
            # We explicitly pass ONLY the core parameters the TE backend expects
            layer_outputs = layer_module(hidden_states, attention_mask)

            # TE layers sometimes return tuples (output, attention_probs)
            if isinstance(layer_outputs, tuple):
                hidden_states = layer_outputs[0]
            else:
                hidden_states = layer_outputs

        # 4. Extract the representation of the entire cell (Position 0 = [CLS])
        cell_embedding = hidden_states[:, 0, :]

        # 5. Push through regression layer for final dosage prediction
        return self.regression_head(cell_embedding)

# Instantiate the model
regression_model = GeneformerForRegression(model_config)
print("✅ Success: 'regression_model' instantiated with Unrolled execution bypass.")

In [ ]:
import pickle
import torch
from torch.utils.data import DataLoader, Dataset

# =====================================================================
# 1. DEFINE CUSTOM SINGLE-CELL DOSAGE DATASET
# =====================================================================
class GoutDosageDataset(Dataset):
    def __init__(self, data_path):
        # Load the serialized tokens and guide count dosages
        with open(data_path, 'rb') as f:
            data = pickle.load(f)
        self.tokens = data['tokens']
        self.dosage = data['dosage']

    def __len__(self):
        return len(self.tokens)

    def __getitem__(self, idx):
        # Package tokens and target dosage (num_features) into tensor structures
        return {
            "input_ids": torch.tensor(self.tokens[idx], dtype=torch.long),
            "target": torch.tensor([self.dosage[idx]], dtype=torch.float)
        }

# =====================================================================
# 2. INSTANTIATE THE DATA CONVEYOR BELT (train_loader)
# =====================================================================
# Point to your saved data path from Phase 3
handover_data_path = 'data/geneformer_tuning_input.pkl'

train_dataset = GoutDosageDataset(handover_data_path)

# Declare 'train_loader' in the global namespace
train_loader = DataLoader(
    train_dataset,
    batch_size=4,       # Process 4 cells at a time to stay safe on memory
    shuffle=True        # Randomize order to optimize backpropagation learning
)

print(f"✅ Success: 'train_loader' successfully initialized with {len(train_dataset)} single-cell samples.")

In [ ]:
import torch
import torch.nn as nn
from torch.optim import AdamW

# 1. Hardware Verification
# TransformerEngine relies heavily on modern GPU architectures (FP8 capability)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"📡 Training environment routed to: {device}")

# Move your custom architecture onto the active hardware
regression_model.to(device)

# 2. Optimization Settings
# We use AdamW (Adam with Weight Decay), which is the standard for Transformers
optimizer = AdamW(regression_model.parameters(), lr=5e-5, weight_decay=0.01)
criterion = nn.MSELoss()  # Mean Squared Error Loss: Measures the prediction gap

# 3. Training Execution Loop
NUM_EPOCHS = 3
print("\n🚀 Initiating Training Loop on 35 Pilot Cells...")

for epoch in range(NUM_EPOCHS):
    regression_model.train()  # Explicitly switch module to 'Training Mode'
    total_epoch_loss = 0.0

    # Iterate through data conveyor belt batches
    for batch_idx, batch in enumerate(train_loader):
        # Safely ship tensors to your GPU
        input_ids = batch["input_ids"].to(device)
        targets = batch["target"].to(device)

        # --- THE GRADIENT DANCE ---
        optimizer.zero_grad()  # 1. Clear previous errors out of memory

        # 2. Forward Pass: Generate continuous dosage prediction
        predictions = regression_model(input_ids=input_ids)

        # 3. Compute error (MSE Scorecard)
        loss = criterion(predictions, targets)

        # 4. Backward Pass: Calculate the mathematical adjustments (Gradients)
        loss.backward()

        # 5. Optimization Step: Tweak the transformer weights
        optimizer.step()

        total_epoch_loss += loss.item()

    avg_loss = total_epoch_loss / len(train_loader)
    print(f"🔹 Epoch [{epoch+1}/{NUM_EPOCHS}] | Average Training MSE Loss: {avg_loss:.4f}")

print("\n🎯 Model Fine-Tuning Sequence Completed Successfully.")

In [ ]:
import os

# 1. Ensure the data directory exists
os.makedirs('data', exist_ok=True)

# 2. Define the save path
save_path = 'data/gout_geneformer_finetuned.pt'

# 3. Extract and save the model's internal learned weights (State Dictionary)
torch.save(regression_model.state_dict(), save_path)

print(f"💾 Milestone Saved: Fine-tuned model checkpoint securely written to {save_path}")